In [2]:
import pybiber as pb
import polars as pl

# ── Config ────────────────────────────────────────────────────────────────────
# Biber/spaCy parquets live at the repo root (python_ML/), one level above this notebook's
# HAP-E_site/ folder -- hence the '../'. The GoEmotions parquet is local (data_processed/).
BIBER_PATH = "../df_biber_hap-e.parquet"   # Biber feature matrix (required)
SPACY_PATH = "../df_spacy_hap-e.parquet"   # token-level spaCy parses (required for per-token tagging)
DROP_FEATURES = ["f_43_type_token", "f_44_mean_word_length"]
N_FACTORS     = 3

# ── Load pre-processed features & parses ─────────────────────────────────────
df_biber = pl.read_parquet(BIBER_PATH).drop(DROP_FEATURES)
df_spacy = pl.read_parquet(SPACY_PATH)
print(f"df_biber shape: {df_biber.shape}")
print(f"df_spacy shape: {df_spacy.shape}")
print(f"Author groups: {df_biber['author'].value_counts()}")

# ── Build analyzer & run MDA ─────────────────────────────────────────────────
analyzer = pb.BiberAnalyzer(df_biber, id_column=True)
print(f"MDA grouping by {len(analyzer.doc_cats)} authors: {analyzer.doc_cats}")
analyzer.mda(n_factors=N_FACTORS)
print(analyzer.mda_summary)


INFO:pybiber.biber_analyzer:Dropping 13 variable(s) with max |r| <= 0.20: ['f_17_agentless_passives', 'f_18_by_passives', 'f_20_existential_there', 'f_22_that_adj_comp', 'f_30_that_obj', 'f_32_wh_obj', 'f_33_pied_piping', 'f_34_sentence_relatives', 'f_36_though', 'f_46_downtoners', 'f_53_modal_necessity', 'f_57_verb_suasive', 'f_66_neg_synthetic']


df_biber shape: (106236, 67)
df_spacy shape: (60672769, 9)
Author groups: shape: (13, 2)
┌──────────────────────┬───────┐
│ author               ┆ count │
│ ---                  ┆ ---   │
│ str                  ┆ u32   │
╞══════════════════════╪═══════╡
│ llama-3-8b           ┆ 8277  │
│ llama-3-70b          ┆ 8281  │
│ gpt-4o               ┆ 8290  │
│ claude-haiku-4-5     ┆ 8276  │
│ gpt-5-mini           ┆ 8279  │
│ gpt-4o-mini          ┆ 8290  │
│ gemma-2-9b-it        ┆ 8288  │
│ llama-3-70b-instruct ┆ 8290  │
│ gemma-2-9b           ┆ 7495  │
│ gemma-2-27b          ┆ 7600  │
│ gemma-2-27b-it       ┆ 8290  │
│ llama-3-8b-instruct  ┆ 8290  │
│ human                ┆ 8290  │
└──────────────────────┴───────┘
MDA grouping by 13 authors: ['claude-haiku-4-5', 'gemma-2-27b', 'gemma-2-27b-it', 'gemma-2-9b', 'gemma-2-9b-it', 'gpt-4o', 'gpt-4o-mini', 'gpt-5-mini', 'human', 'llama-3-70b', 'llama-3-70b-instruct', 'llama-3-8b', 'llama-3-8b-instruct']
shape: (3, 6)
┌──────────┬─────────────┬───────

# HAP-E website JSON — two separate files (Biber + GoEmotions)

Builds the two data files for the style-comparison website. They are **separate, not hybrid** —
each tab reads its own file:

- **`biber_paragraphs.json`** → the *Paragraph Examples* tab. The most Biber-divergent
  `human` vs `AUTHOR_B` pairs **per genre** (the site shows a genre picker), every token tagged
  with the Biber features it matched (MDA over `df_biber_hap-e.parquet` + `df_spacy_hap-e.parquet`).
  Schema:
  `{factor, authorA, authorB, featureLoadings, genres, paragraphsByGenre:{<genre>:[{examples:[{aTokens,bTokens}]}]}}`.
- **`emotions_paragraphs.json`** → the *Emotional Tone* tab. Corpus-wide per-emotion log2
  firing-rate ratios (AI group vs human, all genres) **plus** the most emotion-contrasting
  `human` vs `gemma-2-9b-it` acad pairs, every sentence tagged with its fired emotions. Built by
  `6b.emotions_paragraphs_json.py` (imported below). Schema:
  `{authorA, authorB, emotionMeta, emotionLoadings, emotionRates, paragraphs:[{baseId, examples:[{aSentences,bSentences,aSummary,bSummary}]}]}`.

The last cell (`inject_json.py`) **inlines both files into `HAP-E_Researcher_website.html`**, since
a `file://` page can't `fetch()` local JSON — the page stays one portable file.

**Cell order / dependencies:**

| cell | writes | needs |
|------|--------|-------|
| Biber cell | `biber_paragraphs.json` | cells 1–3 above (df_spacy, analyzer, FEATURE_FILTERS) — heavy |
| Emotion cell | `emotions_paragraphs.json` | only `data_processed/goemotions_sentence_probs.parquet` (run 4a) — light, independent |
| Build step | updates the HTML | both JSON files |

Run **4a** first (produces the GoEmotions parquet), then this notebook top to bottom.

*Scaffolding authored by Claude.*

In [3]:
import re
import polars as pl
import polars.selectors as cs
from pybiber.biber_dict import FEATURES, WORDLISTS

# ── 1. Add the same lead/lag columns that pybiber builds internally ──────────
def _with_lags(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        *(pl.col("dep_rel").shift(i, fill_value="punct").over("doc_id").alias(f"dep_lag_{i}")
          for i in [*range(-3, 0), *range(1, 2)]),
        *(pl.col("lemma").shift(i).over("doc_id").alias(f"lem_lag_{i}")
          for i in range(1, 3)),
        *(pl.col("pos").shift(i).over("doc_id").alias(f"pos_lag_{i}")
          for i in [*range(-4, 0), *range(1, 3)]),
        *(pl.col("tag").shift(i, fill_value="PUNCT").over("doc_id").alias(f"tag_lag_{i}")
          for i in [*range(-3, 0), *range(1, 3)]),
        *(pl.col("token").shift(i).over("doc_id").alias(f"tok_lag_{i}")
          for i in [*range(-3, 0), *range(1, 2)]),
    ])

# ── 2. Add token_tag column (lowercase token + "_" + lowercase tag) ──────────
def _with_token_tag(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .filter((pl.col("token") != " ") & (pl.col("tag") != "_SP"))
        .with_columns(
            pl.when(pl.col("dep_rel") == "punct").then(pl.lit("_punct")).otherwise(pl.col("token")).alias("_t"),
            pl.when(pl.col("dep_rel") == "punct").then(pl.lit("")).otherwise(pl.col("tag")).alias("_tg"),
        )
        .with_columns(
            pl.concat_str([pl.col("_t").str.to_lowercase(), pl.col("_tg").str.to_lowercase()], separator="_").alias("token_tag")
        )
        .drop("_t", "_tg")
    )

# Build annotated token frame once (reused by all lookups)
df_annotated = _with_token_tag(_with_lags(df_spacy))

# ── 3. Precompile regex patterns ──────────────────────────────────────────────
_PAT = {k: re.compile("|".join(v)) for k, v in FEATURES.items()}
def _rx(name):
    return lambda df: df.filter(pl.col("token_tag").str.contains(_PAT[name].pattern))

# ── 4. Filter dispatch table (all 67 features) ───────────────────────────────
FEATURE_FILTERS = {
    # — Regex-based (match against "token_tag" = "word_postag") —
    "f_01_past_tense":              _rx("f_01_past_tense"),
    "f_03_present_tense":           _rx("f_03_present_tense"),
    "f_04_place_adverbials":        _rx("f_04_place_adverbials"),
    "f_05_time_adverbials":         _rx("f_05_time_adverbials"),
    "f_06_first_person_pronouns":   _rx("f_06_first_person_pronouns"),
    "f_07_second_person_pronouns":  _rx("f_07_second_person_pronouns"),
    "f_08_third_person_pronouns":   _rx("f_08_third_person_pronouns"),
    "f_09_pronoun_it":              _rx("f_09_pronoun_it"),
    "f_11_indefinite_pronouns":     _rx("f_11_indefinite_pronouns"),
    "f_20_existential_there":       _rx("f_20_existential_there"),
    "f_24_infinitives":             _rx("f_24_infinitives"),
    "f_33_pied_piping":             _rx("f_33_pied_piping"),
    "f_36_though":                  _rx("f_36_though"),
    "f_37_if":                      _rx("f_37_if"),
    "f_42_adverbs":                 _rx("f_42_adverbs"),
    "f_45_conjuncts":               _rx("f_45_conjuncts"),
    "f_46_downtoners":              _rx("f_46_downtoners"),
    "f_47_hedges":                  _rx("f_47_hedges"),
    "f_48_amplifiers":              _rx("f_48_amplifiers"),
    "f_49_emphatics":               _rx("f_49_emphatics"),
    "f_50_discourse_particles":     _rx("f_50_discourse_particles"),
    "f_52_modal_possibility":       _rx("f_52_modal_possibility"),
    "f_53_modal_necessity":         _rx("f_53_modal_necessity"),
    "f_54_modal_predictive":        _rx("f_54_modal_predictive"),
    "f_55_verb_public":             _rx("f_55_verb_public"),
    "f_56_verb_private":            _rx("f_56_verb_private"),
    "f_57_verb_suasive":            _rx("f_57_verb_suasive"),
    "f_58_verb_seem":               _rx("f_58_verb_seem"),
    "f_59_contractions":            _rx("f_59_contractions"),
    "f_66_neg_synthetic":           _rx("f_66_neg_synthetic"),
    "f_67_neg_analytic":            _rx("f_67_neg_analytic"),
    # — Filter-based (explicit Polars conditions) —
    "f_02_perfect_aspect": lambda df: df.filter(
        (pl.col("lemma") == "have") & pl.col("dep_rel").str.contains("aux")
    ),
    "f_10_demonstrative_pronoun": lambda df: df.filter(
        (pl.col("tag") == "DT")
        & ((pl.col("tag_lag_1").is_null()) | (~pl.col("tag_lag_1").str.contains("^N|^CD|DT")))
        & pl.col("dep_rel").str.contains("nsubj|dobj|pobj")
        & pl.col("token").str.to_lowercase().is_in(WORDLISTS["pronoun_matchlist"])
    ),
    "f_12_proverb_do": lambda df: df.filter(
        (pl.col("lemma") == "do") & ~pl.col("dep_rel").str.contains("aux")
    ),
    "f_13_wh_question": lambda df: df.with_columns(
        pl.int_range(pl.len()).over(["doc_id", "sentence_id"]).alias("_si")
    ).filter(
        pl.col("tag").str.contains("^W") & (pl.col("pos") != "DET")
        & (pl.col("dep_lag_-1") == "aux")
        & ((pl.col("pos_lag_1") == "PUNCT") | (pl.col("pos_lag_2") == "PUNCT") | (pl.col("_si") <= 1))
    ).drop("_si"),
    "f_14_nominalizations": lambda df: df.filter(
        (pl.col("pos") == "NOUN")
        & pl.col("token").str.to_lowercase().str.contains("tion$|tions$|ment$|ments$|ness$|nesses$|ity$|ities$")
        & ~pl.col("token").str.to_lowercase().is_in(WORDLISTS["nominalization_stoplist"])
    ),
    "f_15_gerunds": lambda df: df.filter(
        pl.col("token").str.to_lowercase().str.contains("ing$|ings$")
        & pl.col("dep_rel").str.contains("nsub|dobj|pobj")
        & ~pl.col("token").str.to_lowercase().is_in(WORDLISTS["gerund_stoplist"])
    ),
    "f_16_other_nouns": lambda df: df.filter(
        ((pl.col("pos") == "NOUN") | (pl.col("pos") == "PROPN"))
        & ~pl.col("token").str.contains("-")
        & ~(pl.col("token").str.to_lowercase().str.contains("ing$|ings$")
            & pl.col("dep_rel").str.contains("nsub|dobj|pobj")
            & ~pl.col("token").str.to_lowercase().is_in(WORDLISTS["gerund_stoplist"]))
        & ~((pl.col("pos") == "NOUN")
            & pl.col("token").str.to_lowercase().str.contains("tion$|tions$|ment$|ments$|ness$|nesses$|ity$|ities$")
            & ~pl.col("token").str.to_lowercase().is_in(WORDLISTS["nominalization_stoplist"]))
    ),
    "f_17_agentless_passives": lambda df: df.filter(
        (pl.col("dep_rel") == "auxpass")
        & (pl.col("tok_lag_-2").is_null() | (pl.col("tok_lag_-2") != "by"))
        & (pl.col("tok_lag_-3").is_null() | (pl.col("tok_lag_-3") != "by"))
    ),
    "f_18_by_passives": lambda df: df.filter(
        (pl.col("dep_rel") == "auxpass")
        & ((pl.col("tok_lag_-2") == "by") | (pl.col("tok_lag_-3") == "by"))
    ),
    "f_19_be_main_verb": lambda df: df.filter(
        (pl.col("lemma") == "be")
        & pl.col("tag").is_in(["VBD", "VBP", "VBZ"])
        & ~pl.col("dep_rel").str.contains("aux")
    ),
    "f_21_that_verb_comp": lambda df: df.filter(
        (pl.col("token") == "that") & (pl.col("pos") == "SCONJ") & (pl.col("pos_lag_1") == "VERB")
    ),
    "f_22_that_adj_comp": lambda df: df.filter(
        (pl.col("token") == "that") & (pl.col("pos") == "SCONJ") & (pl.col("pos_lag_1") == "ADJ")
    ),
    # Note: raw matches before doc-level adjustment (f_23 -= f_31 + f_32)
    "f_23_wh_clause": lambda df: df.filter(
        pl.col("tag").str.contains("^W") & (pl.col("token") != "which") & (pl.col("pos_lag_1") == "VERB")
    ),
    "f_25_present_participle": lambda df: df.filter(
        (pl.col("tag") == "VBG")
        & ((pl.col("dep_rel") == "advcl") | (pl.col("dep_rel") == "ccomp"))
        & (pl.col("dep_lag_1") == "punct")
    ),
    "f_26_past_participle": lambda df: df.filter(
        (pl.col("tag") == "VBN")
        & ((pl.col("dep_rel") == "advcl") | (pl.col("dep_rel") == "ccomp"))
        & (pl.col("dep_lag_1") == "punct")
    ),
    "f_27_past_participle_whiz": lambda df: df.filter(
        (pl.col("tag") == "VBN") & (pl.col("dep_rel") == "acl") & (pl.col("pos_lag_1") == "NOUN")
    ),
    "f_28_present_participle_whiz": lambda df: df.filter(
        (pl.col("tag") == "VBG") & (pl.col("dep_rel") == "acl") & (pl.col("pos_lag_1") == "NOUN")
    ),
    "f_29_that_subj": lambda df: df.filter(
        (pl.col("token").str.to_lowercase() == "that")
        & pl.col("dep_rel").str.contains("nsubj")
        & pl.col("tag_lag_1").str.contains("^N|^CD|DT")
    ),
    "f_30_that_obj": lambda df: df.filter(
        (pl.col("token").str.to_lowercase() == "that")
        & pl.col("dep_rel").str.contains("dobj")
        & pl.col("tag_lag_1").str.contains("^N|^CD|DT")
    ),
    "f_31_wh_subj": lambda df: df.filter(
        pl.col("tag").str.contains("^W")
        & (pl.col("lem_lag_2") != "ask") & (pl.col("lem_lag_2") != "tell")
        & (pl.col("tag_lag_1").str.contains("^N|^CD|DT")
           | ((pl.col("pos_lag_1") == "PUNCT") & pl.col("tag_lag_2").str.contains("^N|^CD|DT") & (pl.col("token") == "who")))
        & (pl.col("token") != "that")
        & pl.col("dep_rel").str.contains("nsubj")
    ),
    "f_32_wh_obj": lambda df: df.filter(
        pl.col("tag").str.contains("^W")
        & (pl.col("lem_lag_2") != "ask") & (pl.col("lem_lag_2") != "tell")
        & (pl.col("tag_lag_1").str.contains("^N|^CD|DT")
           | ((pl.col("pos_lag_1") == "PUNCT") & pl.col("tag_lag_2").str.contains("^N|^CD|DT") & (pl.col("token") == "who")))
        & (pl.col("token") != "that")
        & pl.col("dep_rel").str.contains("obj")
    ),
    "f_34_sentence_relatives": lambda df: df.filter(
        (pl.col("token").str.to_lowercase() == "which") & (pl.col("pos_lag_1") == "PUNCT")
    ),
    "f_35_because": lambda df: df.filter(
        (pl.col("token").str.to_lowercase() == "because")
        & (pl.col("tok_lag_-1").str.to_lowercase() != "of")
    ),
    "f_38_other_adv_sub": lambda df: df.filter(
        (pl.col("pos") == "SCONJ") & (pl.col("dep_rel") == "mark")
        & ~pl.col("token").str.to_lowercase().is_in(["because", "if", "unless", "though", "although", "tho"])
        & ~((pl.col("token").str.to_lowercase() == "that") & (pl.col("dep_lag_1") != "ADV"))
    ),
    "f_39_prepositions": lambda df: df.filter(pl.col("dep_rel") == "prep"),
    "f_40_adj_attr": lambda df: df.filter(
        (pl.col("pos") == "ADJ")
        & ((pl.col("pos_lag_-1") == "NOUN") | (pl.col("pos_lag_-1") == "ADJ")
           | ((pl.col("tok_lag_-1") == ",") & (pl.col("pos_lag_-2") == "ADJ")))
        & ~pl.col("token").str.contains("-")
    ),
    "f_41_adj_pred": lambda df: df.filter(
        (pl.col("pos") == "ADJ")
        & ((pl.col("pos_lag_1") == "VERB") | (pl.col("pos_lag_1") == "AUX"))
        & pl.col("lem_lag_1").is_in(WORDLISTS["linking_matchlist"])
        & (pl.col("pos_lag_-1") != "NOUN") & (pl.col("pos_lag_-1") != "ADJ") & (pl.col("pos_lag_-1") != "ADV")
        & ~pl.col("token").str.contains("-")
    ),
    "f_51_demonstratives": lambda df: df.filter(
        pl.col("token").str.to_lowercase().is_in(WORDLISTS["pronoun_matchlist"])
        & (pl.col("dep_rel") == "det")
    ),
    "f_60_that_deletion": lambda df: df.filter(
        pl.col("lemma").is_in(WORDLISTS["verb_matchlist"]) & (pl.col("pos") == "VERB")
        & (
            ((pl.col("dep_lag_-1") == "nsubj") & (pl.col("pos_lag_-2") == "VERB")
             & (pl.col("tag_lag_-1") != "WP") & (pl.col("tag_lag_-2") != "VBG"))
            | ((pl.col("tag_lag_-1") == "DT") & (pl.col("dep_lag_-2") == "nsubj") & (pl.col("pos_lag_-3") == "VERB"))
            | ((pl.col("tag_lag_-1") == "DT") & (pl.col("dep_lag_-2") == "amod")
               & (pl.col("dep_lag_-3") == "nsubj") & (pl.col("pos_lag_-4") == "VERB"))
        )
    ),
    "f_61_stranded_preposition": lambda df: df.filter(
        (pl.col("tag") == "IN") & (pl.col("dep_rel") == "prep")
        & pl.col("tag_lag_-1").str.contains("^[[:punct:]]$")
    ),
    "f_62_split_infinitive": lambda df: df.filter(
        (pl.col("tag") == "TO")
        & (
            ((pl.col("tag_lag_-1") == "RB") & (pl.col("tag_lag_-2") == "VB"))
            | ((pl.col("tag_lag_-1") == "RB") & (pl.col("tag_lag_-2") == "RB") & (pl.col("tag_lag_-3") == "VB"))
        )
    ),
    "f_63_split_auxiliary": lambda df: df.filter(
        pl.col("dep_rel").str.contains("aux")
        & (
            ((pl.col("pos_lag_-1") == "ADV") & (pl.col("pos_lag_-2") == "VERB"))
            | ((pl.col("pos_lag_-1") == "ADV") & (pl.col("pos_lag_-2") == "ADV") & (pl.col("pos_lag_-3") == "VERB"))
        )
    ),
    "f_64_phrasal_coordination": lambda df: df.filter(
        (pl.col("tag") == "CC")
        & (
            ((pl.col("pos_lag_-1") == "NOUN") & (pl.col("pos_lag_1") == "NOUN"))
            | ((pl.col("pos_lag_-1") == "VERB") & (pl.col("pos_lag_1") == "VERB"))
            | ((pl.col("pos_lag_-1") == "ADJ")  & (pl.col("pos_lag_1") == "ADJ"))
            | ((pl.col("pos_lag_-1") == "ADV")  & (pl.col("pos_lag_1") == "ADV"))
        )
    ),
    "f_65_clausal_coordination": lambda df: df.filter(
        (pl.col("tag") == "CC") & (pl.col("dep_rel") != "ROOT")
        & (
            (pl.col("dep_lag_-1") == "nsubj")
            | (pl.col("dep_lag_-2") == "nsubj")
            | (pl.col("dep_lag_-3") == "nsubj")
        )
    ),
    # f_43 (type-token ratio) and f_44 (mean word length) are corpus-level
    # metrics with no individual token attribution — excluded intentionally.
}

_OUTPUT_COLS = ["doc_id", "sentence_id", "token_id", "token", "lemma", "pos", "tag", "dep_rel"]

def get_feature_tokens(feature_name: str, doc_id: str = None) -> pl.DataFrame:
    """Return the token rows from df_spacy that were counted for a Biber feature.

    Parameters
    ----------
    feature_name : str
        Any of the 65 tagged Biber features, e.g. 'f_14_nominalizations'.
        (f_43 and f_44 are excluded — they have no per-token attribution.)
    doc_id : str, optional
        Restrict to a single document.
    """
    if feature_name not in FEATURE_FILTERS:
        raise ValueError(
            f"Unknown feature '{feature_name}'.\nAvailable: {sorted(FEATURE_FILTERS)}"
        )
    result = FEATURE_FILTERS[feature_name](df_annotated)
    if doc_id is not None:
        result = result.filter(pl.col("doc_id") == doc_id)
    keep = [c for c in _OUTPUT_COLS if c in result.columns]
    return result.select(keep)

# ── Example ───────────────────────────────────────────────────────────────────
get_feature_tokens("f_14_nominalizations").head(10)

doc_id,sentence_id,token_id,token,lemma,pos,tag,dep_rel
str,u32,i64,str,str,str,str,str
"""human_acad_0001""",1,3,"""management""","""management""","""NOUN""","""NN""","""compound"""
"""human_acad_0001""",1,31,"""variations""","""variation""","""NOUN""","""NNS""","""pobj"""
"""human_acad_0001""",2,55,"""management""","""management""","""NOUN""","""NN""","""compound"""
"""human_acad_0001""",3,65,"""management""","""management""","""NOUN""","""NN""","""compound"""
"""human_acad_0001""",4,109,"""implementation""","""implementation""","""NOUN""","""NN""","""conj"""
"""human_acad_0001""",4,112,"""management""","""management""","""NOUN""","""NN""","""compound"""
"""human_acad_0001""",5,130,"""management""","""management""","""NOUN""","""NN""","""compound"""
"""human_acad_0001""",6,148,"""Motivation""","""motivation""","""NOUN""","""NN""","""nsubjpass"""
"""human_acad_0001""",6,173,"""participation""","""participation""","""NOUN""","""NN""","""pobj"""


In [6]:
import polars as pl
import json
import re
from pathlib import Path

# Requires cells above to have defined: df_spacy, df_annotated, FEATURE_FILTERS, _PAT, analyzer

# ── Config ───────────────────────────────────────────────────────────────────
FACTOR_NUM  = 1                 # 1, 2, 3 — which MDA factor to rank Biber contrasts on
AUTHOR_A    = "human"           # Style A label
AUTHOR_B    = "gemma-2-9b-it"   # Style B — the LLM the Biber tab contrasts against.
                                # Matches the Emotional Tone tab (also gemma-2-9b-it, set in 6b),
                                # so both website tabs compare human vs the same model.
GENRES      = ["acad"]          # Biber examples are drawn from academic prose only. Any genre with
                                # no human/AUTHOR_B parallel pairs is skipped.
OUTPUT_PATH = "biber_paragraphs.json"   # Biber-only website data (read by the "Paragraph Examples" tab)
N_PAIRS      = 10               # how many contrasting pairs to emit per genre
TARGET_WORDS = 180             # approx word count per Biber sample — roughly one paragraph
DIVERGENT_QUANTILE = 0.60      # "divergent" = contrast at/above this quantile (top ~40% of pairs).
                               # We then spread the N_PAIRS picks evenly across that pool instead of
                               # taking the N single most-extreme pairs — divergent, but not only outliers.
                               # (Thin genres fall back to the top N_PAIRS so they still show N topics.)

factor_col = f"factor_{FACTOR_NUM}"

# ── 0. Detect regex features Polars can't handle (look-around) ───────────────
# Polars' Rust regex engine rejects look-behind / look-ahead. For those Biber feature
# patterns, we run the compiled Python regex on each row instead.
_LOOKAROUND_RE = re.compile(r"\(\?<[=!]|\(\?[=!]")

def _has_lookaround(pattern_str):
    return bool(_LOOKAROUND_RE.search(pattern_str))

try:
    _PAT_REF = _PAT  # defined in the FEATURE_FILTERS cell
except NameError:
    _PAT_REF = {}

_PY_FALLBACK = {
    name: compiled
    for name, compiled in _PAT_REF.items()
    if _has_lookaround(compiled.pattern)
}
if _PY_FALLBACK:
    print(f"Python fallback for {len(_PY_FALLBACK)} feature(s) with look-around regex:")
    for n in sorted(_PY_FALLBACK):
        print(f"  · {n}")

# ── 1. Top 5 positive & 5 negative loadings on selected factor ───────────────
# Used only to score sentence windows for selection. The emitted JSON tags every
# token with ALL tagged Biber features so the front-end can regroup without regenerating.
f_load    = analyzer.mda_loadings.select(["feature", factor_col]).sort(factor_col)
top_feats = f_load.tail(5)["feature"].to_list() + f_load.head(5)["feature"].to_list()

print(f"\nScoring sentence windows by top 5 ± loadings on {factor_col}:")
print("  +", f_load.tail(5)["feature"].to_list())
print("  -", f_load.head(5)["feature"].to_list())

# ── 2. Pick N_PAIRS divergent pairs per genre (spread across the divergent pool) ──
# All docs carry a factor score; we filter to one genre, join parallel human↔AUTHOR_B
# samples on their shared base id, rank by |factor-score difference| (contrast), then
# spread the selection across the divergent pool rather than taking only the extremes.
_all_scores = (
    analyzer.mda_dim_scores
    .select(["doc_id", "doc_cat", factor_col])
    .with_columns(pl.col("doc_id").str.split("_").list.get(1).alias("_genre"))
)

def _base_doi(col, author):
    return col.str.replace(f"^{author}_", "").str.replace(r"@.+$", "")

def select_pairs_for_genre(genre):
    """Return the divergent, spread-selected pair table for one genre (may be empty)."""
    scores = _all_scores.filter(pl.col("_genre") == genre)
    a_scores = (
        scores.filter(pl.col("doc_cat") == AUTHOR_A)
        .with_columns(
            _base_doi(pl.col("doc_id"), AUTHOR_A).alias("doi"),
            pl.col("doc_id").alias("doc_id_a"),
            pl.col(factor_col).alias("f_a"),
        )
        .select(["doi", "doc_id_a", "f_a"])
    )
    b_scores = (
        scores.filter(pl.col("doc_cat") == AUTHOR_B)
        .with_columns(
            _base_doi(pl.col("doc_id"), AUTHOR_B).alias("doi"),
            pl.col("doc_id").alias("doc_id_b"),
            pl.col(factor_col).alias("f_b"),
        )
        .select(["doi", "doc_id_b", "f_b"])
    )
    pairs = (
        a_scores.join(b_scores, on="doi")          # join on the shared base id (parallel sample)
        .with_columns((pl.col("f_a") - pl.col("f_b")).abs().alias("contrast"))
        .sort("contrast", descending=True)
    )
    if pairs.height == 0:
        return pairs
    target_k  = min(N_PAIRS, pairs.height)                    # how many topics we want to show
    cutoff    = pairs["contrast"].quantile(DIVERGENT_QUANTILE)
    divergent = pairs.filter(pl.col("contrast") >= cutoff)    # still sorted by contrast desc
    if divergent.height < target_k:
        # Thin genre: the divergent pool is smaller than we want to display. Fall back to the
        # top target_k most-divergent pairs so the genre still shows up to N_PAIRS topics.
        return pairs.head(target_k)
    # Spread the picks evenly across the divergent pool (most→least divergent), so examples
    # stay contrasting without all being the same handful of outliers. Deterministic.
    n = divergent.height
    if N_PAIRS == 1:
        idx = [0]
    else:
        idx = sorted({round(k * (n - 1) / (N_PAIRS - 1)) for k in range(N_PAIRS)})
    return divergent.with_row_index("_rank").filter(pl.col("_rank").is_in(idx)).drop("_rank")

genre_pairs = {}   # genre -> top_pairs DataFrame (insertion order = display order)
print(f"\nSelecting divergent {AUTHOR_A} vs {AUTHOR_B} pairs per genre on {factor_col}:")
for g in GENRES:
    tp = select_pairs_for_genre(g)
    if tp.height:
        genre_pairs[g] = tp
        print(f"  {g:5s}: {tp.height} pair(s), contrast {tp['contrast'].min():.2f}–{tp['contrast'].max():.2f}")
    else:
        print(f"  {g:5s}: no {AUTHOR_A}/{AUTHOR_B} parallel pairs — skipped")
if not genre_pairs:
    raise ValueError(f"No {AUTHOR_A}/{AUTHOR_B} parallel pairs in any of {GENRES}.")

# ── 3. Pre-filter annotated + spacy frames to just the target docs (all genres) ──
target_doc_ids = sorted({
    d for tp in genre_pairs.values()
    for d in (tp["doc_id_a"].to_list() + tp["doc_id_b"].to_list())
})
df_subset       = df_annotated.filter(pl.col("doc_id").is_in(target_doc_ids))
df_spacy_subset = df_spacy.filter(pl.col("doc_id").is_in(target_doc_ids))
print(f"\nWorking with {df_subset.height:,} annotated tokens across {len(target_doc_ids)} docs "
      f"({len(genre_pairs)} genres)")

def _fast_tokens(feature_name, doc_id):
    """Return the token rows from df_subset that this feature matches for doc_id.

    For regex features whose pattern uses look-around, falls back to applying
    the compiled Python regex in a per-row map (slow but correct)."""
    if feature_name in _PY_FALLBACK:
        doc_rows = df_subset.filter(pl.col("doc_id") == doc_id)
        if doc_rows.is_empty():
            return doc_rows
        pat = _PY_FALLBACK[feature_name]
        mask = doc_rows["token_tag"].map_elements(
            lambda s: bool(pat.search(s)) if s is not None else False,
            return_dtype=pl.Boolean,
        )
        return doc_rows.filter(mask)
    return FEATURE_FILTERS[feature_name](df_subset).filter(pl.col("doc_id") == doc_id)

# ── 4. Per-token Biber tagging ───────────────────────────────────────────────
# For each target doc, build a map (sentence_id, token_id) -> [feature_names].
ALL_TAGGED_FEATURES = list(FEATURE_FILTERS.keys())
print(f"\nTagging tokens with {len(ALL_TAGGED_FEATURES)} Biber features per doc…")

def build_tag_map(doc_id):
    tag_map = {}
    for feat in ALL_TAGGED_FEATURES:
        for row in _fast_tokens(feat, doc_id).iter_rows(named=True):
            key = (row["sentence_id"], row["token_id"])
            tag_map.setdefault(key, []).append(feat)
    return tag_map

def typical_sentence_ids(doc_id, target_words=TARGET_WORDS):
    """Pick a contiguous window of sentences ≳ target_words long that is TYPICAL of the
    document — its feature-density closest to the document's median window-density —
    rather than the densest window. Gives a representative passage, not an outlier."""
    doc_sents = (
        df_spacy_subset
        .filter(pl.col("doc_id") == doc_id)
        .group_by("sentence_id")
        .agg(pl.len().alias("wc"))
        .sort("sentence_id")
    )
    sent_ids = doc_sents["sentence_id"].to_list()
    wc_list  = doc_sents["wc"].to_list()
    if not sent_ids:
        return []
    if sum(wc_list) <= target_words:
        return sent_ids

    frames = [t for t in (_fast_tokens(f, doc_id) for f in top_feats) if not t.is_empty()]
    hit_map = {}
    if frames:
        hits = pl.concat(frames).group_by("sentence_id").agg(pl.len().alias("n"))
        hit_map = dict(zip(hits["sentence_id"].to_list(), hits["n"].to_list()))

    # Enumerate every contiguous window ≳ target_words and its feature density (hits/word).
    candidates = []
    for i in range(len(sent_ids)):
        wc, window = 0, []
        for j in range(i, len(sent_ids)):
            wc += wc_list[j]
            window.append(sent_ids[j])
            if wc >= target_words:
                break
        if wc < target_words:
            continue
        score = sum(hit_map.get(s, 0) for s in window)
        candidates.append((score / wc, window))   # density = feature hits per word
    if not candidates:
        return sent_ids

    # Choose the window whose density is TYPICAL (closest to the median across candidates).
    # Ties break to the earliest window (candidates are in start-index order) — deterministic.
    densities = sorted(d for d, _ in candidates)
    median = densities[len(densities) // 2]
    _, best_window = min(candidates, key=lambda c: abs(c[0] - median))
    return best_window

def tokens_for_doc(doc_id):
    """Return [{t, f}, …] for a representative (typical-density) window of doc_id, in order."""
    sent_ids = typical_sentence_ids(doc_id)
    tag_map  = build_tag_map(doc_id)
    rows = (
        df_spacy_subset
        .filter((pl.col("doc_id") == doc_id) & pl.col("sentence_id").is_in(sent_ids))
        .sort(["sentence_id", "token_id"])
    )
    return [
        {"t": r["token"], "f": tag_map.get((r["sentence_id"], r["token_id"]), [])}
        for r in rows.iter_rows(named=True)
    ]

# ── 5. Build paragraphs, grouped by genre (Biber tokens only) ────────────────
# Schema consumed by the "Paragraph Examples" tab (genre picker + topic buttons):
#   genres            = ["acad", "news", …]                      # display order
#   paragraphsByGenre = { "acad": [ { "examples": [ {
#       "aTokens": [{t, f:[biber…]}, …],  "bTokens": [{t, f:[…]}, …],
#   } ] }, … ], … }
# The per-sentence GoEmotions layer lives in the SEPARATE emotions_paragraphs.json (next cell);
# this file is intentionally Biber-only — no hybrid.
paragraphs_by_genre = {}
for g, tp in genre_pairs.items():
    out = []
    print(f"\n════ genre: {g} ════")
    for row in tp.iter_rows(named=True):
        doc_a, doc_b = row["doc_id_a"], row["doc_id_b"]
        a_tokens = tokens_for_doc(doc_a)
        b_tokens = tokens_for_doc(doc_b)
        out.append({"examples": [{"aTokens": a_tokens, "bTokens": b_tokens}]})
        a_text   = " ".join(t["t"] for t in a_tokens)
        b_text   = " ".join(t["t"] for t in b_tokens)
        a_tagged = sum(1 for t in a_tokens if t["f"])
        b_tagged = sum(1 for t in b_tokens if t["f"])
        print(f"\n── [{row['doi']}]  contrast={row['contrast']:.2f}  ──")
        print(f"  {AUTHOR_A} ({a_tagged}/{len(a_tokens)} tokens tagged):")
        print(f"    {a_text}")
        print(f"  {AUTHOR_B} ({b_tagged}/{len(b_tokens)} tokens tagged):")
        print(f"    {b_text}")
    paragraphs_by_genre[g] = out

# ── 6. Biber feature loadings on the selected factor ─────────────────────────
feature_loadings = dict(zip(
    analyzer.mda_loadings["feature"].to_list(),
    analyzer.mda_loadings[factor_col].to_list(),
))

# ── 7. Write the Biber JSON ──────────────────────────────────────────────────
output_blob = {
    "factor":            factor_col,
    "authorA":           AUTHOR_A,
    "authorB":           AUTHOR_B,
    "featureLoadings":   feature_loadings,          # Biber feature -> loading on this factor (corpus-wide)
    "genres":            list(paragraphs_by_genre.keys()),
    "paragraphsByGenre": paragraphs_by_genre,
}
Path(OUTPUT_PATH).write_text(json.dumps(output_blob, indent=2))
total_pairs = sum(len(v) for v in paragraphs_by_genre.values())
print(f"\n✓ Wrote {total_pairs} paragraph pair(s) across {len(paragraphs_by_genre)} genres "
      f"({', '.join(paragraphs_by_genre)}) to {OUTPUT_PATH}")
print(f"  factor={factor_col}, authors={AUTHOR_A} vs {AUTHOR_B}, "
      f"{len(feature_loadings)} Biber loadings embedded (Biber-only, no emotion layer)")


Python fallback for 1 feature(s) with look-around regex:
  · f_48_amplifiers

Scoring sentence windows by top 5 ± loadings on factor_1:
  + ['f_65_clausal_coordination', 'f_56_verb_private', 'f_42_adverbs', 'f_06_first_person_pronouns', 'f_59_contractions']
  - ['f_16_other_nouns', 'f_39_prepositions', 'f_40_adj_attr', 'f_27_past_participle_whiz', 'f_14_nominalizations']

Selecting divergent human vs gemma-2-9b-it pairs per genre on factor_1:
  acad : 3 pair(s), contrast 3.45–17.41
  news : 3 pair(s), contrast 5.18–25.44
  blog : 3 pair(s), contrast 6.74–29.83
  fic  : 3 pair(s), contrast 6.04–23.95
  spok : 3 pair(s), contrast 7.51–34.00
  tvm  : 3 pair(s), contrast 9.85–35.56

Working with 18,938 annotated tokens across 36 docs (6 genres)

Tagging tokens with 65 Biber features per doc…

════ genre: acad ════

── [acad_0606]  contrast=17.41  ──
  human (138/199 tokens tagged):
    Also , while it is true that soil characteristics are correlated , they are not so highly correlated that

In [5]:
# ── Emotional Tone website data — emotions_paragraphs.json ───────────────────
# The emotion counterpart to the Biber cell above. SELF-CONTAINED: it needs only
# data_processed/goemotions_sentence_probs.parquet (a 4a output) — NOT df_spacy / analyzer —
# so it can run even without the heavy Biber load above. The logic lives in
# 6b.emotions_paragraphs_json.py (single source of truth, also runnable as a standalone CLI);
# we import its build function and write the JSON here so this notebook generates BOTH website
# files. emotions_paragraphs.json is emotion-only (per-sentence GoEmotions + corpus loadings) —
# no Biber tokens, no hybrid.
import importlib.util
import json
from pathlib import Path

_spec = importlib.util.spec_from_file_location(
    "emo6b", Path("6b.emotions_paragraphs_json.py").resolve())
emo6b = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(emo6b)

emotions_blob = emo6b.build_emotions_blob()              # reads the GoEmotions parquet, returns the dict
Path("emotions_paragraphs.json").write_text(json.dumps(emotions_blob, indent=2))
print("✓ Wrote emotions_paragraphs.json\n")
emo6b.report(emotions_blob)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/user/Github/Code_ML/python_ML/HAP-E_site/6b.emotions_paragraphs_json.py'

In [ ]:
# ── Build step: inline both JSON files into the website HTML ─────────────────
# The site is opened straight from disk (file://), where the browser refuses to fetch() local
# .json files. So this step embeds biber_paragraphs.json and emotions_paragraphs.json directly
# into HAP-E_Researcher_website.html (replacing the `const paragraphsJson = {…}` and
# `const emotionsJson = {…}` blobs in place). Run it after regenerating either JSON; the page
# stays a single portable file you can double-click. Logic lives in inject_json.py.
import importlib.util
from pathlib import Path

_spec = importlib.util.spec_from_file_location("inject_json", Path("inject_json.py").resolve())
inject_json = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(inject_json)
inject_json.main()


  inlined biber_paragraphs.json  ->  const paragraphsJson  (54,828 chars)
  inlined emotions_paragraphs.json  ->  const emotionsJson  (30,734 chars)
Updated HAP-E_Researcher_website.html
